In [1]:
import sys
sys.path.append('../libs')

%load_ext autoreload
%autoreload 2

import ios
from tools_comparison import ToolsComparison

Total combinations of features: 1023


In [2]:
VERSION = 3
DIR_DATA = f'../../data/v{VERSION}/'
DIR_GT = ios.path_join(DIR_DATA, 'annotations')

RESULTS = f'../../results/v{VERSION}/gender'
DIR_TOOL_W = ios.path_join(RESULTS, 'tool_w')
DIR_TOOL_R = ios.path_join(RESULTS, 'tool_r')
DIR_TOOL_N = ios.path_join(RESULTS, 'tool_n')

In [3]:
VERSION = 3
RANDOM_SEED = 1

VAR_X = ['mean', 'median', 'mode']
DIR = f'../../results/v{VERSION}/gender/predictions/comparison/rs*/<ML>/_<VAR_X>/*'
METRIC_CLASS = 'test_auc'
METRIC_REG = 'test_rmse'
ASC_CLASS = False
ASC_REG = True

# Best Model

## Classification

In [10]:
dir_class = DIR.replace('<ML>', 'classification').replace("/*","/cut_*/*")
df_class_mean = ios.load_json_files_to_dataframe(dir_class.replace('<VAR_X>', 'mean'), include_fn=True)
df_class_median = ios.load_json_files_to_dataframe(dir_class.replace('<VAR_X>', 'median'), include_fn=True)
df_class_mode = ios.load_json_files_to_dataframe(dir_class.replace('<VAR_X>', 'mode'), include_fn=True)

df_class = ios.pd.concat([df_class_mean, df_class_median, df_class_mode], ignore_index=True)
df_class.loc[:,'rs'] = df_class.fn.apply(lambda v: int(v.split('/rs')[-1].split('/')[0]))
df_class = df_class.query("rs != 12345").copy()

cols_class = ['y','features_id','strategy_id','features','nfeatures','test_accuracy','test_f1','test_precision','test_recall','test_auc']
best_id_class = df_class[METRIC_CLASS].idxmax()

best_id_class, df_class.loc[best_id_class].features, df_class.loc[best_id_class].test_f1, df_class.loc[best_id_class].f_name  

(151,
 ['gender_n',
  'gender_nw',
  'mean_ngender_rn',
  'mean_ngender_rw',
  'mean_ngender_nw'],
 0.8825396825396825,
 'mean_607_1')

In [11]:
### @deprecated: when we were using 'male' for W and single char.
# Best without W
# filtered_df = df_class[df_class['features'].apply(lambda lst: all('w' not in item for item in lst))]
# filtered_df.loc[filtered_df[METRIC_CLASS].idxmax()]

df_class.loc[df_class[METRIC_CLASS].idxmax(), cols_class]


y                                                       gender_mean
features_id                                                     607
strategy_id                                                       1
features          [gender_n, gender_nw, mean_ngender_rn, mean_ng...
nfeatures                                                         5
test_accuracy                                                 0.875
test_f1                                                     0.88254
test_precision                                                  0.9
test_recall                                                0.872086
test_auc                                                   0.970959
Name: 151, dtype: object

## Regression

In [12]:
dir_reg = DIR.replace('<ML>', 'regression')
df_reg_mean = ios.load_json_files_to_dataframe(dir_reg.replace('<VAR_X>', 'mean'), include_fn=True)
df_reg_median = ios.load_json_files_to_dataframe(dir_reg.replace('<VAR_X>', 'median'), include_fn=True)
df_reg_mode = ios.load_json_files_to_dataframe(dir_reg.replace('<VAR_X>', 'mode'), include_fn=True)

df_reg = ios.pd.concat([df_reg_mean, df_reg_median, df_reg_mode], ignore_index=True)
df_reg.loc[:,'rs'] = df_reg.fn.apply(lambda v: int(v.split('/rs')[-1].split('/')[0]))
df_reg = df_reg.query("rs != 12345").copy()

cols_reg = ['y','features_id','features','nfeatures', 'train_eval','test_r2','test_rmse','rs']
best_id_reg = df_reg[METRIC_REG].idxmin()

best_id_reg, df_reg.loc[best_id_reg].features, df_reg.loc[best_id_reg].test_rmse, df_reg.loc[best_id_reg].f_name

(926,
 ['ngender_r',
  'ngender_n',
  'gender_w',
  'single_char',
  'mean_ngender_rn',
  'mean_ngender_nw',
  'mean_ngender_rnw'],
 0.09357271367688783,
 'mean_894')

In [13]:
### @deprecated: when we were using 'male' for W and single char.
# # Best without W
# filtered_df = df_reg[df_reg['features'].apply(lambda lst: all('w' not in item for item in lst))]
# filtered_df.loc[filtered_df[METRIC_REG].idxmin()]

df_reg.loc[df_reg[METRIC_REG].idxmin(), cols_reg]

y                                                           mean
features_id                                                  894
features       [ngender_r, ngender_n, gender_w, single_char, ...
nfeatures                                                      7
train_eval                                               0.09727
test_r2                                                 0.850307
test_rmse                                               0.093573
rs                                                             2
Name: 926, dtype: object

___
# OLD
Code below is deprecated. To predict all names use `scrips/batch_gender_prediction.py`

## Data

### Given Names

In [12]:
tc = ToolsComparison(dir_tool_w=DIR_TOOL_W, dir_tool_r=DIR_TOOL_R, dir_tool_n=DIR_TOOL_N, dir_gt=DIR_GT)
tc.load_data()
tc.compare_unique_chinese_given_names()
tc.df_ch_genders.head(2)

,ngender_r,gender_w,0,1,2,3,4,5,6,7,...,std_ngender_rnw,std_ngender_rn,std_ngender_rw,std_ngender_nw,mode,mode_ngender_rnw,mode_ngender_rn,mode_ngender_rw,mode_ngender_nw,gender_nw
givenname,,,,,,,,,,,,,,,,,,,,,
俊池,0.30315,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.161985,0.035063,0.151575,0.186638,NaN,0.30315,0.30315,0.30315,0.373277,male
建军,0.14830,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.125561,0.079598,0.074150,0.153748,NaN,0.14830,0.14830,0.14830,0.307495,male


In [14]:
tc.df_ch_genders.shape

(39713, 40)

### Dictionary Chinese Characters

In [15]:
FN_DICT_NAMSOR = f'../../results/v{VERSION}/chinese_gender/tool_n/character_dictionary_withEng_name_gender.csv'
df_dict_gender = ios.load_csv(FN_DICT_NAMSOR)
df_dict_gender.head(2)

,character,pinyin_hanziDB,pinyin_qxk,meaning,pinyin_hanziDB_qxk,pinyin_name,pinyin_givenname,pinyin_familyname,pinyin,tone,...,NC,apiVersion,script,id,firstName,lastName,likelyGender,genderScale,score,probabilityCalibrated
0,一,yī,yī,yī\t1:数目字，最小的正整数。2:相同。～样|～起|～致|统～|说法不～。3:全；满。～...,yī,yī,yī,yī,yī,1,...,3.0,namsor_api_2.1.1,HAN,3f22e65c-c4e7-4888-a6ca-79194d6a4dbe,一,NaN,male,-0.453237,3.152061,0.726618
1,乙,yǐ,yǐ,yǐ\t1:天干的第二位，常用作顺序的第二。2:中国民族音乐音阶的一级，乐谱上用作记音符号，...,yǐ,yǐ,yǐ,yǐ,yǐ,3,...,2.8,namsor_api_2.1.1,HAN,760f4156-6964-471c-961f-399162d20874,乙,NaN,female,0.972487,10.789135,0.986244


In [16]:
df_dict_gender.shape

(8096, 35)

## Prediction

In [21]:
# predict gender using best classification model
fn_class_model = df_class.loc[best_id_class].f_name
path = DIR.replace('<ML>', 'classification')
path = path.replace('<VAR_X>', fn_class_model.split("_")[0])
fn_class_model = ios.path_join(path, f"{fn_class_model}.cbm")
fn_class_model 


'../../results/v2/chinese_gender/comparison/classification/_mean/mean_230_1.cbm'

In [22]:
# predict gender using best regression model
fn_reg_model = df_reg.loc[best_id_reg].f_name
path = DIR.replace('<ML>', 'regression')
path = path.replace('<VAR_X>', fn_reg_model.split("_")[0])
fn_reg_model = ios.path_join(path, f"{fn_reg_model}.cbm")
fn_reg_model

'../../results/v2/chinese_gender/comparison/regression/_mean/mean_50.cbm'

In [42]:
from catboost import CatBoostRegressor
model = CatBoostRegressor()
model.load_model('../../results/v2/chinese_gender/comparison/regression/_mean/mean_50.cbm', format="cbm")
model

['ngender_r', 'gender_nw', 'mean_ngender_rnw']